<a href="https://colab.research.google.com/github/WuilderOsio1198/Senalesysistemas/blob/main/Ejercicio_de_Simulacion_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Descargue 10 segundos de la canción de su preferencia desde youtube, y generé un filtro pasabanda (el usuario debe poder definir las frecuencias de corte) para cada uno de los filtros descritos (el usuario también debe poder fijar los parámetros de diseño de cada filtro). Compare los resultados de los filtros estudiados en este cuaderno tipo IIR para diseño Butterworth, Chebyshev 1, Chebyshev 2, Bessel y Elíptico.

In [ ]:

#! pip install youtube-dl
!python3 -m pip install --force-reinstall https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz

     - 2.9 MB 9.4 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for yt-dlp: filename=yt_dlp-2025.12.8-py3-none-any.whl size=3038170 sha256=a1314f46f22efbb13ed5b2c22338f4b7f65ed92ed310fb13aa154fecf2e5463b
  Stored in directory: /tmp/pip-ephem-wheel-cache-vth_jere/wheels/b6/70/13/8d2d11b326f983030b72df6408392d5c1b3bc27a9db8b7c5b0
Successfully built yt-dlp


In [7]:

link="https://www.youtube.com/watch?v=a_wBQrUdcvk&list=RDa_wBQrUdcvk&start_radio=1"
!yt-dlp --extract-audio -o "audio" --audio-format mp3 {link}

[youtube] Extracting URL: https://www.youtube.com/watch?v=a_wBQrUdcvk
[youtube] a_wBQrUdcvk: Downloading webpage
[youtube] a_wBQrUdcvk: Downloading android sdkless player API JSON
[youtube] a_wBQrUdcvk: Downloading web safari player API JSON
[youtube] a_wBQrUdcvk: Downloading m3u8 information
[info] a_wBQrUdcvk: Downloading 1 format(s): 251
ERROR: unable to download video data: HTTP Error 403: Forbidden


In [5]:

!pip install soundfile

In [ ]:
import soundfile as sf # para instalar pip install soundfile
#lee archivos wav
nombre_out = "audio.mp3"
x, fs = sf.read(nombre_out)

# read speech signal from file
print('Frecuencia de muestreo %.2f[Hz]\naudio %s' % (fs,nombre_out))

LibsndfileError: Error opening 'audio.mp3': System error.

In [ ]:

x.shape[0]/fs #segundos de la canción

In [ ]:

xpro = x.copy() #copiar archivos para procesar
ti = 30
tf = 40
xs = xpro[int(ti*fs):int((tf*fs)),:]

In [ ]:

#Para escuchar el trozo de audio con el que se va a trabajar:
#No usar un trozo muy grande o el entorno se puede desconectar
from IPython.display import Audio
Audio([xs[:,1], xs[:,0]],rate=fs)

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.patches import Circle

from scipy.signal import butter as bw
from scipy.signal import freqz_zpk
from scipy.signal import cheby1, cheby2, ellip, bessel, freqz
from scipy.signal import lfilter, filtfilt

import soundfile as sf
from IPython.display import Audio


# Función para respuesta en frecuencia

def plot_freq_response(filter_name, w, h, N): #Mostrar la respuesta en frecuencia del filtro
  fig = plt.figure()
  ax1 = fig.add_subplot(1, 1, 1)
  ax1.set_title(filter_name +' Digital filter frequency response, order= ' + str(N))
  ax1.plot(w, 20 * np.log10(abs(h)), 'b')
  ax1.set_ylabel('Amplitude [dB]', color='b')
  ax1.set_xlabel('Frequency [Hz]')
  ax1.grid()
  ax2 = ax1.twinx()
  angles = np.unwrap(np.angle(h))
  ax2.plot(w, angles, 'g')
  ax2.set_ylabel('Angle [radians]', color='g')
  plt.axis('tight')
  plt.show()

#Función para mostrar polos y ceros
def show_zp(z, p, title= 'Z-plane'): # Mostrar la ubicación de los polos y los zeros
  ax = plt.gca()

  ax.plot(np.real(z), np.imag(z), 'bo', fillstyle='none', ms = 10)
  ax.plot(np.real(p), np.imag(p), 'rx', fillstyle='none', ms = 10)
  unit_circle = Circle((0,0), radius=1, fill=False,
                        color='black', ls='solid', alpha=0.9)
  ax.add_patch(unit_circle)
  ax.axvline(0, color='0.7')
  ax.axhline(0, color='0.7')

  plt.title(title)
  plt.xlabel(r'Re{$z$}')
  plt.ylabel(r'Im{$z$}')
  plt.axis('equal')
  plt.xlim((-2, 2))
  plt.ylim((-2, 2))
  plt.grid()


Aquí es donde el usuario puede cambiar orden y frecuencias de corte del pasabanda:

In [ ]:
# PARÁMETROS

# Frecuencias de corte del pasabanda (en Hz)
f1 = 2000   # corte inferior
f2 = 5000   # corte superior

Wn = [f1, f2]      # banda pasante
filt = 'bandpass'  # tipo del filtro pasabanda

print("Banda pasante:", Wn, "Hz")


In [ ]:
# FILTRO BUTTERWORTH PASABANDA

from scipy.signal import butter as bw
from scipy.signal import freqz_zpk

# Primero diseño para ver polos y ceros (output='zpk')
N = 4  # orden del filtro Butterworth
out = 'zpk'

zeros, poles, gain = bw(N, Wn, btype=filt, output=out, fs=fs)
w, h = freqz_zpk(zeros, poles, gain, fs=fs) # respuesta en frecuencia

show_zp(zeros, poles, title='Butterworth - Polos y Ceros')
plot_freq_response('Butterworth', w, h, N)

# Ahora lo diseño en forma (b,a) para filtrar la señal de audio
b_bw, a_bw = bw(N, Wn, btype=filt, output='ba', fs=fs)

# Usamos filtfilt para no tener retardo de fase
xf_bw = filtfilt(b_bw, a_bw, xs, axis=0)

Audio([xf_bw[:,1], xf_bw[:,0]], rate=fs)  # Resultado Butterworth


In [ ]:
# FILTRO CHEBYSHEV TIPO I (IIR)

from scipy.signal import cheby1

N = 4            # orden Chebyshev I
ripple = 1       # rizado en banda pasante en dB
out = 'zpk'

zeros_c1, poles_c1, gain_c1 = cheby1(N, rp=ripple, Wn=Wn, btype=filt, output=out, fs=fs)

w_c1, h_c1 = freqz_zpk(zeros_c1, poles_c1, gain_c1, fs=fs)

show_zp(zeros_c1, poles_c1, title='Chebyshev I - Polos y Ceros')
plot_freq_response('Chebyshev1', w_c1, h_c1, N)

# Para filtrar el audio lo diseñamos en forma (b,a)
b_c1, a_c1 = cheby1(N, rp=ripple, Wn=Wn, btype=filt, output='ba', fs=fs)
xf_c1 = filtfilt(b_c1, a_c1, xs, axis=0)
print()
Audio([xf_c1[:,1], xf_c1[:,0]], rate=fs)  # Resultado Chebyshev I

In [ ]:
# FILTRO CHEBYSHEV TIPO 2 (IIR)

from scipy.signal import cheby2
from scipy.signal import freqz

N = 4               # orden del filtro Chebyshev II
ripple_stop = 30    # atenuación mínima en banda de rechazo (dB)
out = 'ba'          # no regresa los polos y ceros, sino la función de transferencia

num_c2, den_c2 = cheby2(N, rs=ripple_stop, Wn=Wn, btype=filt, output=out, fs=fs)
w_c2, h_c2 = freqz(num_c2, den_c2, fs=fs)

plot_freq_response('Chebyshev2', w_c2, h_c2, N)

# Filtrar el audio
xf_c2 = filtfilt(num_c2, den_c2, xs, axis=0)
print()
Audio([xf_c2[:,1], xf_c2[:,0]], rate=fs)  # Resultado Chebyshev II


In [ ]:
# FILTRO ELÍPTICO (CAUER) IIR

from scipy.signal import ellip

N = 4              # orden elíptico
ripple_pass = 1    # rizado en banda pasante (dB)
ripple_stop = 40   # atenuación en banda de rechazo (dB)
out = 'ba'

num_el, den_el = ellip(N, rp=ripple_pass, rs=ripple_stop, Wn=Wn, btype=filt, output=out, fs=fs)
w_el, h_el = freqz(num_el, den_el, fs=fs)

plot_freq_response('Elliptic', w_el, h_el, N)

# Filtrar el audio
xf_el = filtfilt(num_el, den_el, xs, axis=0)
print()
Audio([xf_el[:,1], xf_el[:,0]], rate=fs)  # Resultado Elíptico


In [ ]:
# FILTRO BESSEL PASABANDA

from scipy.signal import bessel

N = 4
normalization = 'mag'  # Ajusta la frecuencia crítica según la magnitud

num_be, den_be = bessel(N, Wn=Wn, btype=filt, norm=normalization, output='ba', fs=fs)
w_be, h_be = freqz(num_be, den_be, fs=fs)

plot_freq_response('Bessel', w_be, h_be, N)

# Filtrar el audio
xf_be = filtfilt(num_be, den_be, xs, axis=0)
print()
Audio([xf_be[:,1], xf_be[:,0]], rate=fs)  # Resultado Bessel


2. Consulte en qué consiste el método de diseño de filtros FIR por ventaneo . Realice un cuadro comparativo de las ventajas y desventajas de los filtros IIR y los FIR. Nota: Recuerde que un filtro FIR utiliza solamente raíces tipo ceros, es decir que $a_0=1$, y $a_k=0$ $\forall k\in\{1,2,\dots\}$.

El diseño por ventaneo sigue tres pasos principales:

## 1. Elegir la respuesta al impulso ideal $$ h_d[n] $$

Cada tipo de filtro (pasa bajas, pasa altas, pasa banda o elimina banda) posee una respuesta ideal que cumple exactamente la frecuencia de corte.

Ejemplo:  
Para un filtro pasa bajas ideal, la respuesta al impulso es infinita y de tipo sinc:

$$
h_d[n] = \frac{\sin(\omega_c (n - M))}{\pi (n - M)}
$$

Esta respuesta ideal es infinita, por lo que no puede implementarse directamente en un filtro real.


## 2. Truncar la respuesta ideal usando una ventana

Para obtener un filtro finito, se multiplica la respuesta ideal $$ h_d[n] $$ por una ventana: $$ w[n] $$

$$
h[n] = h_d[n] \cdot w[n]
$$

La ventana:

- limita la longitud del filtro  
- suaviza los bordes, reduciendo el ripple en frecuencia  
- controla el ancho del lóbulo principal y la atenuación de los lóbulos laterales  

### Ventanas típicas

| **Ventana**   | **Atenuación aprox.** | **Características** |
|---------------|------------------------|----------------------|
| Rectangular   | -21 dB                 | Mejor resolución, mayor ripple |
| Hann          | -44 dB                 | Balanceada y comúnmente usada |
| Hamming       | -53 dB                 | Menos ripple que Hann |
| Blackman      | -74 dB                 | Excelente atenuación fuera de banda |
| Kaiser        | Ajustable              | Permite controlar ripple y transición |



## 3. Ajustar el orden del filtro \( N \)

El orden del filtro determina:

- el ancho de la banda de transición  
- el nivel de ripple permitido  
- la calidad general del filtro  

Una fórmula aproximada para varias ventanas es:

$$
N \approx \frac{A - 8}{2.285\, \Delta\omega}
$$

Donde:

- \( A \): atenuación deseada (en dB)  
- {\Delta\omega}: ancho de la banda de transición  
